# 02 - Video Game Metadata: Output Data Analysis & Explorer

Unified notebook for viewing, searching, and analyzing the final cleaned video game dataset (`cleaned_games`).

### Contents
1. **Load Data & Overview**: Fast load via Parquet with SQLite connection.
2. **Interactive Search & Filter**: Search by title, platform, genre, year, and rating.
3. **Missingness & Schema Analysis**: Field completeness across canonical columns.
4. **Platform Analysis**: Coverage, top platforms, and cross-platform titles.
5. **Name & Title Analysis**: Lengths, character sets, and sequel/version patterns.
6. **Summary Text Analysis**: Word count distributions and descriptions.
7. **Genre Analysis**: Frequency breakdown, co-occurrences, and translation verification.
8. **Release Timeline Analysis**: Historical distributions, decade aggregation, and date consistency.
9. **Developer & Publisher Analysis**: Leading studios and publishers.
10. **Players & Cooperative Play**: Player count distributions and co-op support.
11. **Rating Distributions**: Critic (`rating`) vs community (`user_rating`) analysis and top-rated games.
12. **Direct SQLite Querying**: Indexed SQL lookups against `output/cleaned_games.db`.
13. **Data Quality Invariants**: Automated verification of dataset integrity constraints.

In [ ]:
import sys
from pathlib import Path
import sqlite3
import re
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual configuration
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 1000)

# Paths
DATA_DIR = Path('../output')
PARQUET_PATH = DATA_DIR / 'cleaned_games.parquet'
DB_PATH = DATA_DIR / 'cleaned_games.db'
CSV_PATH = DATA_DIR / 'cleaned_games.csv'

print('Environment and libraries initialized.')


## 1. Load Data & Quick Overview

Loads directly from the high-performance Parquet export (`cleaned_games.parquet`).

In [ ]:
print(f'Loading {PARQUET_PATH}...')
df = pd.read_parquet(PARQUET_PATH)
print(f'✓ Loaded {len(df):,} games across {df["platform"].nunique():,} platforms.')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB')
df.head(5)


## 2. Interactive Search & Filtering Utilities

Reusable functions for interactive queries during development and inspection.

In [ ]:
COLS = ['name', 'platform', 'release_year', 'genres', 'developer', 'publisher', 'rating', 'user_rating']

def search_games(query: str, platform: str = None, min_year: int = None, max_year: int = None, limit: int = 25) -> pd.DataFrame:
    '''Search games by title substring or regex (case-insensitive).'''
    mask = df['name'].str.contains(query, case=False, na=False, regex=True)
    if platform:
        mask &= df['platform'].str.contains(platform, case=False, na=False)
    if min_year is not None:
        mask &= (df['release_year'] >= min_year)
    if max_year is not None:
        mask &= (df['release_year'] <= max_year)
    return df.loc[mask, [c for c in COLS if c in df.columns]].head(limit)

def filter_by_genre(genre: str, platform: str = None, min_rating: float = None, limit: int = 25) -> pd.DataFrame:
    '''Filter games by genre substring with optional platform and rating filters.'''
    mask = df['genres'].str.contains(genre, case=False, na=False)
    if platform:
        mask &= df['platform'].str.contains(platform, case=False, na=False)
    if min_rating is not None:
        mask &= (df['rating'] >= min_rating) | (df['user_rating'] >= min_rating)
    return df.loc[mask, [c for c in COLS if c in df.columns]].sort_values(by=['rating', 'user_rating'], ascending=False).head(limit)

print('Interactive search functions registered.')


In [ ]:
# Example search: Metroid titles
search_games('^Metroid', limit=10)


In [ ]:
# Example filter: High-rated RPGs
filter_by_genre('Role-Playing|RPG', min_rating=9.0, limit=10)


## 3. Missingness & Field Completeness Analysis

In [ ]:
missingness = df.isnull().sum()
missing_pct = (missingness / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Dtype': df.dtypes.astype(str),
    'Non-Null': df.notnull().sum(),
    'Missing Count': missingness,
    'Missing %': missing_pct,
    'Unique': df.nunique()
}).sort_values('Missing %', ascending=True)
missing_df


In [ ]:
# Plot field completeness
fig, ax = plt.subplots(figsize=(10, 6))
(100 - missing_df['Missing %']).plot(kind='barh', ax=ax, color='teal')
ax.set_xlabel('Completeness Percentage (%)')
ax.set_title('Field Completeness Across Canonical Schema')
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()


## 4. Platform Analysis

In [ ]:
platform_counts = df['platform'].value_counts()
print(f'Total unique canonical platforms: {len(platform_counts)}')
print(f'Single-game platforms: {(platform_counts == 1).sum()}')

# Top 25 platforms plot
fig, ax = plt.subplots(figsize=(12, 7))
platform_counts.head(25).plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_xlabel('Game Count')
ax.set_title('Top 25 Platforms by Title Count')
plt.tight_layout()
plt.show()


In [ ]:
# Multi-platform games distribution
titles_per_game = df.groupby('name')['platform'].nunique()
print(f'Unique game titles: {len(titles_per_game):,}')
print(f'Exclusive titles (1 platform): {(titles_per_game == 1).sum():,} ({(titles_per_game == 1).mean()*100:.1f}%)')
print(f'Multi-platform titles (>= 2): {(titles_per_game >= 2).sum():,} ({(titles_per_game >= 2).mean()*100:.1f}%)')
print('\nGames released on the most platforms:')
print(titles_per_game.nlargest(10))


## 5. Name & Title Quality Analysis

In [ ]:
name_lengths = df['name'].str.len()
print(f'Name length: min={name_lengths.min()}, max={name_lengths.max()}, mean={name_lengths.mean():.1f}, median={name_lengths.median():.1f}')

# Check for residual region tags
region_pattern = r'\b(USA|PAL|Japan|Europe|Asia|World)\b'
games_with_region_bracket = df[df['name'].str.contains(r'\([A-Za-z0-9, ]+\)', regex=True, na=False)]
print(f'Titles with parenthetical tags: {len(games_with_region_bracket):,} (e.g. subtitles or edition tags)')

# Shortest game titles
short_names = df[df['name'].str.len() <= 2][['name', 'platform', 'release_year']].drop_duplicates('name')
print(f'Distinct titles with <= 2 characters: {len(short_names)}')
short_names.head(10)


## 6. Summary & Description Text Analysis

In [ ]:
df['summary_word_count'] = df['summary'].apply(lambda x: len(x.split()) if isinstance(x, str) else 0)
has_summary = df['summary_word_count'] > 0
print(f'Games with summaries: {has_summary.sum():,} ({has_summary.mean()*100:.1f}%)')
print(df.loc[has_summary, 'summary_word_count'].describe().round(1))

# Distribution of summary length
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df.loc[has_summary & (df['summary_word_count'] <= 250), 'summary_word_count'], bins=50, ax=ax, color='darkorange')
ax.set_xlabel('Word Count')
ax.set_title('Summary Word Count Distribution (<= 250 words)')
plt.tight_layout()
plt.show()


## 7. Genre Analysis

In [ ]:
# Explode comma-separated genres
all_genres = df['genres'].dropna().str.split(',').explode().str.strip()
all_genres = all_genres[all_genres != '']
genre_counts = all_genres.value_counts()

print(f'Total unique normalized genres: {len(genre_counts)}')

# Top 20 genres
fig, ax = plt.subplots(figsize=(12, 6))
genre_counts.head(20).plot(kind='barh', ax=ax, color='mediumpurple')
ax.invert_yaxis()
ax.set_xlabel('Frequency')
ax.set_title('Top 20 Most Frequent Normalized Genres')
plt.tight_layout()
plt.show()


In [ ]:
# Verify no untranslated Portuguese remnants
pt_remnants = df['genres'].str.contains(r'\b(?:Ação|Plataforma|Esporte|Estratégia|Simulação|Corrida|Pilotagem)\b', regex=True, na=False).sum()
print(f'Portuguese genre remnants count: {pt_remnants} (Target: 0)')
assert pt_remnants == 0, 'Foreign genre remnants found!'


## 8. Release Timeline & Date Consistency Analysis

In [ ]:
valid_years = df.loc[df['release_year'].notnull(), 'release_year'].astype(int)
print(f'Release year coverage: {len(valid_years):,} ({len(valid_years)/len(df)*100:.1f}%)')
print(f'Year range: {valid_years.min()} to {valid_years.max()} (Median: {valid_years.median():.0f})')

# Distribution by year (1970 - 2025)
fig, ax = plt.subplots(figsize=(12, 4))
sns.histplot(valid_years[(valid_years >= 1970) & (valid_years <= 2025)], bins=56, ax=ax, color='royalblue')
ax.set_xlabel('Release Year')
ax.set_title('Releases Per Year (1970 - 2025)')
plt.tight_layout()
plt.show()


In [ ]:
# Check Date vs Year consistency
has_both = df['release_date'].notnull() & df['release_year'].notnull()
year_diff = (df.loc[has_both, 'release_date'].dt.year != df.loc[has_both, 'release_year']).sum()
print(f'Release date vs release_year contradictions: {year_diff} (Target: 0)')
assert year_diff == 0, 'Contradictions between release_date and release_year found!'


## 9. Developer & Publisher Analysis

In [ ]:
# Explode developers & publishers
devs = df['developer'].dropna().str.split(',').explode().str.strip()
pubs = df['publisher'].dropna().str.split(',').explode().str.strip()

top_devs = devs[devs != ''].value_counts().head(15)
top_pubs = pubs[pubs != ''].value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top_devs.plot(kind='barh', ax=axes[0], color='crimson')
axes[0].invert_yaxis()
axes[0].set_title('Top 15 Developers')

top_pubs.plot(kind='barh', ax=axes[1], color='darkslategray')
axes[1].invert_yaxis()
axes[1].set_title('Top 15 Publishers')
plt.tight_layout()
plt.show()


## 10. Players & Cooperative Play Analysis

In [ ]:
print(f'Non-null players count: {df["players"].notna().sum():,} ({df["players"].notna().mean()*100:.1f}%)')
print(f'Non-null cooperative flag: {df["cooperative"].notna().sum():,} ({df["cooperative"].notna().mean()*100:.1f}%)')
print(f'Cooperative games: {(df["cooperative"] == True).sum():,}')

# Player counts distribution (<= 8 players)
player_counts = df['players'].dropna().value_counts().head(8)
print('\nPlayer count distribution:')
print(player_counts)


## 11. Rating Distributions & Critical Reception

Unifies critic ratings (`rating`) and community ratings (`user_rating`) on a standardized 0.0 - 10.0 scale.

In [ ]:
print('Critic Rating (rating) summary:')
print(df['rating'].describe().round(2))
print('\nUser Rating (user_rating) summary:')
print(df['user_rating'].describe().round(2))

# Distribution plots
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['rating'].dropna(), bins=30, ax=axes[0], color='forestgreen', kde=True)
axes[0].set_title('Critic Rating Distribution (0.0 - 10.0)')

sns.histplot(df['user_rating'].dropna(), bins=30, ax=axes[1], color='dodgerblue', kde=True)
axes[1].set_title('User Rating Distribution (0.0 - 10.0)')
plt.tight_layout()
plt.show()


In [ ]:
# Top 15 highest-rated games with both critic and user scores
both = df[df['rating'].notnull() & df['user_rating'].notnull()].copy()
both['composite_score'] = ((both['rating'] + both['user_rating']) / 2).round(2)
top_15 = both.sort_values('composite_score', ascending=False)[['name', 'platform', 'release_year', 'rating', 'user_rating', 'composite_score']].head(15)
top_15


## 12. Direct SQL Querying via SQLite (`cleaned_games.db`)

The SQLite export provides indices on `name`, `platform`, `name + platform`, and `release_year` for instant querying.

In [ ]:
conn = sqlite3.connect(DB_PATH)

def run_query(sql: str) -> pd.DataFrame:
    '''Execute SQL query against output/cleaned_games.db.'''
    return pd.read_sql_query(sql, conn)

# Query 1: Game count and average rating by decade
sql_decades = '''
SELECT 
    (release_year / 10) * 10 AS decade,
    COUNT(*) AS total_games,
    ROUND(AVG(rating), 2) AS avg_critic_rating,
    ROUND(AVG(user_rating), 2) AS avg_user_rating
FROM cleaned_games
WHERE release_year >= 1970 AND release_year <= 2025
GROUP BY decade
ORDER BY decade DESC;
'''
run_query(sql_decades)


In [ ]:
# Query 2: Multi-platform franchise comparison
sql_mario = '''
SELECT name, platform, release_year, rating, user_rating
FROM cleaned_games
WHERE name LIKE 'Super Mario%' AND rating >= 8.5
ORDER BY release_year ASC
LIMIT 10;
'''
run_query(sql_mario)


## 13. Data Quality Invariant Verification

Runs automated integrity checks established in the Definition of Done.

In [ ]:
print('='*70)
print('DATA QUALITY INVARIANT VERIFICATION')
print('='*70)

# Check 1: Rating bounds
rating_in_bounds = (df['rating'].dropna().between(0.0, 10.0)).all()
user_rating_in_bounds = (df['user_rating'].dropna().between(0.0, 10.0)).all()
print(f'1. Rating in [0.0, 10.0]: {rating_in_bounds} (min={df["rating"].min()}, max={df["rating"].max()})')
print(f'2. User rating in [0.0, 10.0]: {user_rating_in_bounds} (min={df["user_rating"].min()}, max={df["user_rating"].max()})')

# Check 2: Date-year contradictions
has_both = df['release_date'].notnull() & df['release_year'].notnull()
contradictions = (df.loc[has_both, 'release_date'].dt.year != df.loc[has_both, 'release_year']).sum()
print(f'3. Date vs year contradictions: {contradictions} (Expected: 0)')

# Check 3: Foreign genre remnants
foreign_genres = df['genres'].str.contains(r'\b(?:Ação|Plataforma|Esporte|Estratégia|Simulação|Corrida|Pilotagem)\b', regex=True, na=False).sum()
print(f'4. Untranslated Portuguese genres: {foreign_genres} (Expected: 0)')

# Check 4: Platform coverage
platform_count = df['platform'].nunique()
print(f'5. Canonical platforms mapped: {platform_count} (Expected: >= 200)')

# Check 5: Total rows
print(f'6. Cleaned dataset rows: {len(df):,} (Expected: >= 400,000)')

assert rating_in_bounds and user_rating_in_bounds, 'Rating out of bounds!'
assert contradictions == 0, 'Date contradictions exist!'
assert foreign_genres == 0, 'Foreign genres found!'
assert platform_count >= 200, 'Platform count below expectation!'
assert len(df) >= 400000, 'Row count below expectation!'

print('='*70)
print('✓ ALL QUALITY INVARIANTS SATISFIED!')
print('='*70)
